# TFLite Export — ASL Sign Language Model
Standalone notebook. **No need to re-run training notebooks.**  
Loads best model (`EfficientNetB0 fine-tuned`) directly from Google Drive and exports TFLite variants for deployment on Raspberry Pi.

## 0. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Imports & Dataset

In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf

# Download test split from Kaggle (for INT8 calibration)
if not os.path.exists("sign_mnist_test.csv") and not os.path.exists("sign_mnist_test/sign_mnist_test.csv"):
    os.environ['KAGGLE_USERNAME'] = "abdullahashiry"
    os.environ['KAGGLE_KEY']      = "KGAT_331632d901a6cb7a05431b55135bd8c2"
    !pip install -q kaggle
    !kaggle datasets download -d datamunge/sign-language-mnist --unzip

test_path = ("sign_mnist_test/sign_mnist_test.csv"
             if os.path.exists("sign_mnist_test/sign_mnist_test.csv")
             else "sign_mnist_test.csv")

test_df = pd.read_csv(test_path)
x_test  = test_df.drop("label", axis=1).values.reshape(-1, 28, 28, 1).astype("float32") / 255.0
print(f"Test set loaded: {x_test.shape}")

Dataset URL: https://www.kaggle.com/datasets/datamunge/sign-language-mnist
License(s): CC0-1.0
100% 62.6M/62.6M [00:00<00:00, 179MB/s]

Test set loaded: (7172, 28, 28, 1)


## 2. Load Trained Model

In [4]:
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess

def fix_lambda_layers(model, preprocess_fn):
    lambdas = [l for l in model.layers if type(l).__name__ == 'Lambda']
    if len(lambdas) >= 1:
        lambdas[0].function = tf.image.grayscale_to_rgb
    if len(lambdas) >= 2:
        lambdas[1].function = preprocess_fn

MODEL_PATH = "/content/drive/MyDrive/CV552_SignLanguage/models/efficientnetb0_finetuned.keras"
model = tf.keras.models.load_model(
    MODEL_PATH, compile=False, safe_mode=False,
    custom_objects={'preprocess_input': eff_preprocess}
)
fix_lambda_layers(model, eff_preprocess)
model.summary()

Model: "efficientnetb0_head"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lambda_4 (Lambda)               │ (None, 28, 28, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing_4 (Resizing)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_5 (Lambda)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ ?                      │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 25)             │         3,225 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,216,764 (16.09 MB)

 Trainable params: 2,204,713 (8.41 MB)

 Non-trainable params: 2,012,051 (7.68 MB)

## 3. Convert to TFLite (Float32)

In [6]:

model.build((None, 28, 28, 1))

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("asl_model.tflite", "wb") as f:
    f.write(tflite_model)
print("Saved: asl_model.tflite")

Saved artifact at '/tmp/tmpada90po0'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='input_layer_12')
Output Type:
  TensorSpec(shape=(None, 25), dtype=tf.float32, name=None)
Captures:
  135992441647376: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  135992441649296: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  135992457194256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135992457197136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135992457195792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135992457197904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135992457198480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135992457198672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135992457198096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135992457197328: TensorSpec(shape=(), dtype=tf.resource, name=Non

## 4. Convert to TFLite (INT8 Quantized — smaller + faster on Pi)

In [7]:
def representative_dataset():
    for i in range(min(200, len(x_test))):
        yield [x_test[i:i+1]]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type  = tf.float32
converter_int8.inference_output_type = tf.float32

tflite_model_int8 = converter_int8.convert()

with open("asl_model_int8.tflite", "wb") as f:
    f.write(tflite_model_int8)
print("Saved: asl_model_int8.tflite")

Saved artifact at '/tmp/tmpjz5fs9p6'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='input_layer_12')
Output Type:
  TensorSpec(shape=(None, 25), dtype=tf.float32, name=None)
Captures:
  135992441647376: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  135992441649296: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  135992457194256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135992457197136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135992457195792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135992457197904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135992457198480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135992457198672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135992457198096: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135992457197328: TensorSpec(shape=(), dtype=tf.resource, name=Non

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Saved: asl_model_int8.tflite


## 5. Sanity Check — Float32 Model

In [8]:
interpreter = tf.lite.Interpreter(model_path="asl_model.tflite")
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Input shape  expected:", input_details[0]['shape'])   # [1, 28, 28, 1]
print("Output shape expected:", output_details[0]['shape'])  # [1, 25]

dummy = np.zeros((1, 28, 28, 1), dtype=np.float32)
interpreter.set_tensor(input_details[0]['index'], dummy)
interpreter.invoke()
out = interpreter.get_tensor(output_details[0]['index'])
print("Output (dummy):", out)

Input shape  expected: [ 1 28 28  1]
Output shape expected: [ 1 25]
Output (dummy): [[0.00064821 0.00225352 0.01291967 0.00929684 0.00490703 0.01727871
  0.00772077 0.01420943 0.00095643 0.00077103 0.32836542 0.0013634
  0.00424816 0.00295181 0.00303443 0.0469006  0.00567435 0.14531517
  0.00613049 0.00219576 0.25366044 0.02414555 0.00192217 0.10203964
  0.00109097]]


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## 6. Model Size Comparison

In [9]:
size_keras = os.path.getsize(MODEL_PATH) / 1e6
size_fp32  = os.path.getsize("asl_model.tflite") / 1e6
size_int8  = os.path.getsize("asl_model_int8.tflite") / 1e6

print(f"Model Size Comparison:")
print(f"  Original .keras : {size_keras:.2f} MB")
print(f"  TFLite FP32     : {size_fp32:.2f} MB")
print(f"  TFLite INT8     : {size_int8:.2f} MB")

Model Size Comparison:
  Original .keras : 35.38 MB
  TFLite FP32     : 16.71 MB
  TFLite INT8     : 5.08 MB


## 7. Copy Models to Drive (optional)

In [10]:
import shutil

DRIVE_OUT = "/content/drive/MyDrive/CV552_SignLanguage/tflite"
os.makedirs(DRIVE_OUT, exist_ok=True)

shutil.copy("asl_model.tflite",      os.path.join(DRIVE_OUT, "asl_model.tflite"))
shutil.copy("asl_model_int8.tflite", os.path.join(DRIVE_OUT, "asl_model_int8.tflite"))
print(f"Copied both .tflite files to {DRIVE_OUT}")

Copied both .tflite files to /content/drive/MyDrive/CV552_SignLanguage/tflite
